# SAHA · Spiker / Yorumcu sesi eğitimi

Bu defter, **SAHA Ses Kaydı** sayfasında kaydedilen cümlelerden oyunun kullanacağı yeni bir ses modeli üretir.
Mevcut Türkçe Piper sesi (Fahrettin, `medium`) temel alınır ve senin kayıtlarınla ince ayar yapılır.
Çıkan dosya oyundaki şimdiki seslerle **aynı biçimdedir** (`.onnx` + `.onnx.json`).

**Hazırlık (bir kez)**
1. Kayıt sayfasında bir rolün cümlelerini bitir ve **Paketi indir**'e dokun (`saha-ses-spiker-…zip`).
2. Google Drive'da **`SAHA-ses`** adında bir klasör aç, zip dosyasını oraya yükle.
3. Bu defterde üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.

**Çalıştırma**
- Hücreleri yukarıdan aşağıya sırayla çalıştır (her hücrenin solundaki ▶).
- 1. hücrede rolü seç: `spiker` ya da `yorumcu`. Her rol ayrı eğitilir.
- Eğitim uzun sürer (birkaç saat). Bağlantı koparsa ya da süre biterse sorun değil: eğitim düzenli olarak Drive'a yedeklenir;
  defteri yeniden açıp hücreleri **aynı rolle** baştan çalıştırınca kaldığı yerden devam eder.
- Son hücreler sesi dinletir ve oyuna konacak dosyaları `SAHA-ses/cikti` klasörüne yazar. O iki dosyayı sohbete gönder, oyuna ben bağlarım.

In [ ]:
#@title 1) Ayarlar
ROL = "spiker"  #@param ["spiker", "yorumcu"]
#@markdown Toplam eğitim turu (epoch). ~25-30 dk kayıt için 1500 iyi bir başlangıç. Kalite yetmezse sonradan artırılabilir.
EPOCH = 1500  #@param {type:"integer"}
#@markdown T4 GPU için 16. "CUDA out of memory" hatası alırsan 12 ya da 8 yap.
BATCH = 16  #@param {type:"integer"}
#@markdown Eğitim Drive'a kaç dakikada bir yedeklensin
YEDEK_DK = 20  #@param {type:"integer"}
KLASOR = "/content/drive/MyDrive/SAHA-ses"
SES_ADI = f"saha_{ROL}"
print("Rol:", ROL, "| ses adı:", SES_ADI, "| epoch:", EPOCH, "| batch:", BATCH)

In [ ]:
#@title 2) Drive'ı bağla ve ekran kartını denetle
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(KLASOR, exist_ok=True)
g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if g.returncode != 0 or not g.stdout.strip():
    raise SystemExit("Ekran kartı (GPU) yok! Üst menüden: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU")
print("GPU:", g.stdout.strip())

In [ ]:
#@title 3) Eğitim yazılımını kur (3-6 dakika)
import subprocess
BETIK = r'''
set -e
apt-get -qq update > /dev/null && apt-get -qq install -y build-essential cmake ninja-build > /dev/null
cd /content
[ -d piper1-gpl ] || git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git
cd piper1-gpl
pip -q install -e '.[train]'
pip -q install onnxruntime huggingface_hub
./build_monotonic_align.sh > /dev/null
python3 setup.py build_ext --inplace > /dev/null 2>&1
python3 -c "import piper.train, piper; print('Kurulum tamam')"
'''
r = subprocess.run(BETIK, shell=True, executable="/bin/bash", capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode != 0:
    print(r.stderr[-3000:]); raise SystemExit("Kurulum başarısız - bu çıktıyı sohbete gönder.")

In [ ]:
#@title 4) Kayıt paketini aç ve denetle
import glob, zipfile, wave, shutil, os
zipler = sorted(glob.glob(f"{KLASOR}/saha-ses-{ROL}-*.zip"), key=os.path.getmtime)
if not zipler:
    raise SystemExit(f"{KLASOR} klasöründe saha-ses-{ROL}-….zip bulunamadı. Kayıt sayfasından indirdiğin paketi Drive'da SAHA-ses klasörüne yükle.")
PAKET = zipler[-1]
VERI = f"/content/veri_{ROL}"
shutil.rmtree(VERI, ignore_errors=True)
zipfile.ZipFile(PAKET).extractall(VERI)
satir, toplam, eksik = [], 0.0, 0
for s in open(f"{VERI}/metadata.csv", encoding="utf-8").read().splitlines():
    if "|" not in s: continue
    ad, metin = s.split("|", 1)
    yol = f"{VERI}/wavs/{ad}"
    if not os.path.exists(yol): eksik += 1; continue
    with wave.open(yol) as w:
        toplam += w.getnframes() / w.getframerate()
    satir.append(f"{ad}|{metin}")
open(f"{VERI}/metadata.csv", "w", encoding="utf-8").write("\n".join(satir) + "\n")
print(f"Paket: {os.path.basename(PAKET)}")
print(f"{len(satir)} kayıt, toplam {toplam/60:.1f} dakika ses" + (f" ({eksik} eksik dosya atlandı)" if eksik else ""))
if toplam < 15 * 60:
    print("UYARI: 15 dakikanın altında kayıt az; ses tanınır olur ama kalitesi düşük kalabilir. Daha çok cümle kaydedilirse sonuç iyileşir.")

In [ ]:
#@title 5) Temel Türkçe sesi indir (Fahrettin, medium)
from huggingface_hub import HfApi, hf_hub_download
api, bulundu = HfApi(), None
for rev in ["main", "refs/pr/4", "refs/pr/5"]:
    try:
        dosyalar = api.list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset", revision=rev)
    except Exception as e:
        continue
    c = sorted(f for f in dosyalar if f.startswith("tr/tr_TR/fahrettin/medium/") and f.endswith(".ckpt"))
    if c:
        bulundu = (rev, c[-1]); break
if not bulundu:
    raise SystemExit("Temel Türkçe kontrol noktası bulunamadı (Hugging Face erişimi?). Bu mesajı sohbete gönder.")
TEMEL = hf_hub_download("rhasspy/piper-checkpoints", bulundu[1], repo_type="dataset", revision=bulundu[0], local_dir="/content/temel")
print("Temel ses:", bulundu[1], "| sürüm:", bulundu[0])

In [ ]:
#@title 6) EĞİT (saatler sürer; durursa aynı rolle hücreleri baştan çalıştır, kaldığı yerden devam eder)
import subprocess, time, glob, os, shutil, re, sys
KOK = f"/content/egitim_{ROL}"
YEDEK = f"{KLASOR}/egitim_{ROL}"
os.makedirs(YEDEK, exist_ok=True)
DEVAM = f"{YEDEK}/last.ckpt"
cmd = ["python3", "-m", "piper.train", "fit",
       "--data.voice_name", f"tr_TR-{SES_ADI}-medium",
       "--data.csv_path", f"{VERI}/metadata.csv",
       "--data.audio_dir", f"{VERI}/wavs",
       "--model.sample_rate", "22050",
       "--data.espeak_voice", "tr",
       "--data.cache_dir", f"/content/onbellek_{ROL}",
       "--data.config_path", f"{YEDEK}/config.json",
       "--data.batch_size", str(BATCH),
       "--data.num_test_examples", "4",
       "--model.mos_metric", "none",
       "--trainer.max_epochs", str(EPOCH),
       "--trainer.default_root_dir", KOK]
if os.path.exists(DEVAM):
    print("Önceki eğitim bulundu, kaldığı yerden devam ediliyor.")
    cmd += ["--ckpt_path", DEVAM]
else:
    print("Yeni eğitim başlıyor (temel: Fahrettin).")
    cmd += ["--model.warmstart_ckpt", TEMEL]

def yedekle():
    L = sorted(glob.glob(f"{KOK}/**/checkpoints/last.ckpt", recursive=True), key=os.path.getmtime)
    if not L: return False
    gecici = DEVAM + ".yaziliyor"
    shutil.copy2(L[-1], gecici); os.replace(gecici, DEVAM)
    return True

log = open(f"/content/egitim_{ROL}.log", "w")
p = subprocess.Popen(cmd, cwd="/content/piper1-gpl", stdout=log, stderr=subprocess.STDOUT)
t0 = son_yedek = time.time(); son_ep, ep0 = None, None
try:
    while p.poll() is None:
        time.sleep(30)
        try:
            metin = open(f"/content/egitim_{ROL}.log", errors="ignore").read()[-20000:]
            m = re.findall(r"Epoch (\d+)", metin)
            if m:
                ep = int(m[-1])
                if ep0 is None: ep0 = ep
                if ep != son_ep:
                    son_ep = ep; gecen = time.time() - t0
                    hiz = gecen / max(1, ep - ep0) if ep > ep0 else None
                    kalan = f" · kalan ~{(EPOCH - ep) * hiz / 3600:.1f} saat" if hiz else ""
                    print(f"epoch {ep}/{EPOCH}{kalan}", flush=True)
        except Exception:
            pass
        if time.time() - son_yedek > YEDEK_DK * 60 and yedekle():
            son_yedek = time.time(); print("  ↳ Drive'a yedeklendi", flush=True)
except KeyboardInterrupt:
    print("Durduruluyor, son durum yedekleniyor…")
    p.terminate(); p.wait(60)
finally:
    yedekle()
    log.close()
print("Eğitim süreci bitti (çıkış kodu %s)." % p.returncode)
if p.returncode not in (0, None, -15):
    print(open(f"/content/egitim_{ROL}.log", errors="ignore").read()[-3000:])

In [ ]:
#@title 7) Oyuna konacak dosyaları üret (ONNX + int8)
import subprocess, json, os, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType
CIKTI = f"{KLASOR}/cikti"; os.makedirs(CIKTI, exist_ok=True)
ckpt = f"{KLASOR}/egitim_{ROL}/last.ckpt"
ad = f"tr_TR-{SES_ADI}-medium"
tam = f"/content/{ad}.onnx"
r = subprocess.run(["python3", "-m", "piper.train.export_onnx", "--checkpoint", ckpt, "--output-file", tam], cwd="/content/piper1-gpl", capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:]); raise SystemExit("ONNX dışa aktarma başarısız")
quantize_dynamic(tam, f"{CIKTI}/{ad}.int8.onnx", weight_type=QuantType.QUInt8)
shutil.copy2(tam, f"{CIKTI}/{ad}.onnx")
cfg = json.load(open(f"{KLASOR}/egitim_{ROL}/config.json", encoding="utf-8"))
cfg.setdefault("espeak", {})["voice"] = "tr"
cfg.setdefault("inference", {"noise_scale": 0.667, "length_scale": 1, "noise_w": 0.8})
json.dump(cfg, open(f"{CIKTI}/{ad}.onnx.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
for f in sorted(os.listdir(CIKTI)):
    if f.startswith(ad): print(f, f"{os.path.getsize(os.path.join(CIKTI, f))/1048576:.1f} MB")

In [ ]:
#@title 8) Dinle (eğitimde olmayan cümleler)
import wave, io
from piper import PiperVoice
from IPython.display import Audio, display
DENEME = ["Ve gol! Yıldırım kaleciyi çaresiz bıraktı! Skor iki bir!",
          "Orta sahada top el değiştirdi, Kalkan hızlı bir kontra başlatıyor.",
          "Bu dakikalarda soğukkanlı kalan takım maçı alır.",
          "Kaleci uzandı ve topu kornere çeldi!"]
for dosya in [f"{CIKTI}/{ad}.int8.onnx", f"{CIKTI}/{ad}.onnx"]:
    v = PiperVoice.load(dosya, config_path=f"{CIKTI}/{ad}.onnx.json")
    print(os.path.basename(dosya))
    for c in DENEME:
        b = io.BytesIO()
        with wave.open(b, "wb") as w: v.synthesize_wav(c, w)
        print("  ", c); display(Audio(b.getvalue(), rate=22050))
print("İki dosya da iyi sesliyorsa .int8.onnx ve .onnx.json dosyalarını sohbete gönder. int8 bozuk geliyorsa tam .onnx dosyasını gönder.")